# CS 553 - Neural Networks Project 1

This notebooks contains code that aims to mimic the results of the MedMNIST V2 paper. We will reproduce and train a ResNet-18 model on the VessleMNIST dataset utilizing 3D convolutions. The module we aim to use is pytorch.

## Image Preprocessing

In [11]:
#import packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch 
from medmnist import VesselMNIST3D
from torch.utils.data import Dataset
from acsconv.converters import ACSConverter, Conv3dConverter
import torchvision.models as models 
from torch.utils.data import DataLoader

In [2]:
#Import VesselMNIST training dataset 
train_dataset = VesselMNIST3D(split="train", download=True)
train_dataset

Dataset VesselMNIST3D of size 28 (vesselmnist3d)
    Number of datapoints: 1335
    Root location: /Users/jonathanphan/.medmnist
    Split: train
    Task: binary-class
    Number of channels: 1
    Meaning of labels: {'0': 'vessel', '1': 'aneurysm'}
    Number of samples: {'train': 1335, 'val': 191, 'test': 382}
    Description: The VesselMNIST3D is based on an open-access 3D intracranial aneurysm dataset, IntrA, containing 103 3D models (meshes) of entire brain vessels collected by reconstructing MRA images. 1,694 healthy vessel segments and 215 aneurysm segments are generated automatically from the complete models. We fix the non-watertight mesh with PyMeshFix and voxelize the watertight mesh with trimesh into 28×28×28 voxels. We split the source dataset with a ratio of 7:1:2 into training, validation and test set.
    License: CC BY 4.0

In [3]:
#import VessleMNIST val dataset
val_dataset = VesselMNIST3D(split="val", download= True)
val_dataset

Dataset VesselMNIST3D of size 28 (vesselmnist3d)
    Number of datapoints: 191
    Root location: /Users/jonathanphan/.medmnist
    Split: val
    Task: binary-class
    Number of channels: 1
    Meaning of labels: {'0': 'vessel', '1': 'aneurysm'}
    Number of samples: {'train': 1335, 'val': 191, 'test': 382}
    Description: The VesselMNIST3D is based on an open-access 3D intracranial aneurysm dataset, IntrA, containing 103 3D models (meshes) of entire brain vessels collected by reconstructing MRA images. 1,694 healthy vessel segments and 215 aneurysm segments are generated automatically from the complete models. We fix the non-watertight mesh with PyMeshFix and voxelize the watertight mesh with trimesh into 28×28×28 voxels. We split the source dataset with a ratio of 7:1:2 into training, validation and test set.
    License: CC BY 4.0

In [4]:
#import VessleMNIST testing dataset
test_dataset = VesselMNIST3D(split="test", download= True)
test_dataset

Dataset VesselMNIST3D of size 28 (vesselmnist3d)
    Number of datapoints: 382
    Root location: /Users/jonathanphan/.medmnist
    Split: test
    Task: binary-class
    Number of channels: 1
    Meaning of labels: {'0': 'vessel', '1': 'aneurysm'}
    Number of samples: {'train': 1335, 'val': 191, 'test': 382}
    Description: The VesselMNIST3D is based on an open-access 3D intracranial aneurysm dataset, IntrA, containing 103 3D models (meshes) of entire brain vessels collected by reconstructing MRA images. 1,694 healthy vessel segments and 215 aneurysm segments are generated automatically from the complete models. We fix the non-watertight mesh with PyMeshFix and voxelize the watertight mesh with trimesh into 28×28×28 voxels. We split the source dataset with a ratio of 7:1:2 into training, validation and test set.
    License: CC BY 4.0

In [5]:
#Look at the first few samples of the dataset
for i in range(5):
    sample_image, sample_label = train_dataset[i]
    print(f'sample image', sample_image, 'sample_label', sample_label)

sample image [[[[0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   ...
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]]

  [[0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   ...
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]]

  [[0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   ...
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]]

  ...

  [[0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   ...
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]]

  [[0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   ...
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]]

  [[0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   ...
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 

In [6]:
sample_image, sample_label = train_dataset[0]
sample_image

array([[[[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]],

        [[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]],

        [[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]],

        ...,

        [[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
    

In [7]:
#Create class that expands the image channel from 1 -> 3
#Data augmentation and normalization
class ExpandedVesselMNIST(Dataset):
    def __init__(self, base_dataset, is_training = True):
        self.base_dataset = base_dataset
        self.is_training = is_training
    
    def __len__(self):
        return len(self.base_dataset)
    
    def __getitem__(self, idx):
        image, label = self.base_dataset[idx]
        tensor_image = torch.from_numpy(image).float()

        if self.is_training:
            rand_factor = torch.rand(1).item()
            tensor_image = tensor_image * rand_factor
        else:
            tensor_image = tensor_image * 0.5

        expanded_image = tensor_image.repeat(3, 1, 1, 1)
        return expanded_image, torch.tensor(label).long()

In [14]:
#Expand image channels for training dataset 
mod_train_set = ExpandedVesselMNIST(train_dataset, is_training=True)
mod_val_set = ExpandedVesselMNIST(val_dataset, is_training=False)
mod_test_set = ExpandedVesselMNIST(test_dataset, is_training=False)

batch_size = 32
train_loader = DataLoader(
    mod_train_set,
    batch_size=32,
    shuffle = True,
    num_workers=2,
    pin_memory=True)

val_loader = DataLoader(
    mod_val_set,
    batch_size= batch_size,
    shuffle = False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    mod_test_set,
    batch_size= batch_size,
    shuffle = False,
    num_workers=2,
    pin_memory=True
)



# Model Architecture and Training: ResNet-18 with 3D Convolutions

In [9]:
#Import resnet18 utilizing an untrained neural network 
resnet18 = models.resnet18(pretrained=False)
#Convert model into 3D convolutional neural network
model_3d = Conv3dConverter(resnet18)

/Users/jonathanphan/anaconda3/envs/myenv/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/jonathanphan/anaconda3/envs/myenv/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


In [ ]:
#Change output nodes to 2 for binary classification task
num_classes = 2
model_3d.fc = torch.nn.Linear(model_3d.fc.in_features, num_classes)

In [ ]:
#Cross Entropy Loss for classification tasks, adam optimizer to match with paper specification
criterion = torch.nn.CrossEntropyLoss()
#Pass through model parameters to optimizers, pass starting learning rate of 0.001
optimizer = torch.optim.Adam(model_3d.parameters(), lr=0.001)

In [ ]:
#Run for 100 epochs
num_epochs = 100
#For each of the epochs
for epoch in range(num_epochs):
    #Set the model to training mode
    model_3d.train()
    #Load each batch per epoch, batch size is 32
    for inputs, labels in train_loader:
        #zero out the gradients per session
        optimizer.zero_grad()
        #Plug in inputs to the NN and get output
        outputs = model_3d(inputs)
        #Feed actual and predicted values into loss function
        loss = criterion(outputs, labels)
        #Backpropogate to compute derivatives
        loss.backward()
        #Updated model weights
        optimizer.step()
    print(f"Epoch{epoch+1}, Loss: {loss.item()}")


